# Lesson 9: エフェクトの原理

**コンパニオンノートブック** — 詳しい解説は本文 Lesson 9 を参照してください。

## セットアップ

In [ ]:
# --- 最初に1回だけ実行 ---
import sys
try:
    import google.colab
    !pip install -q japanize-matplotlib
    !git clone -q https://github.com/ggszk/simple-sound-programming.git
    sys.path.append('/content/simple-sound-programming')
except ImportError:
    sys.path.append('..')

from audio_lib.notebook import setup_environment
setup_environment()

In [ ]:
!pip install -q pedalboard

## このレッスンで学ぶこと

- リバーブ、ディレイ、コーラスがどのような仕組みで音を加工しているかを理解する
- ディレイラインとコムフィルタの原理を学ぶ
- コンプレッサーによるダイナミクス処理の基本を学ぶ
- エフェクトチェーンの順序が音に与える影響を理解する


## 9.2 エフェクトを体験する

In [ ]:
# pedalboard のインストール（Colab の場合）
# !pip install pedalboard

In [ ]:
import numpy as np
from IPython.display import display
from audio_lib import sine_wave, adsr, AudioSignal
from audio_lib.notebook import play_sound, apply_effect

def create_test_sound(freq=440, duration=2.0):
    """テスト用ピアノ風の音"""
    wave = sine_wave(freq, duration)
    env = adsr(duration, attack=0.01, decay=0.3, sustain=0.4, release=0.5)
    return AudioSignal(wave.data * env.data, 44100)

dry = create_test_sound()
display(play_sound(dry, "ドライ音（エフェクトなし）"))

### リバーブを体験する

In [ ]:
from pedalboard import Reverb

reverb = Reverb(room_size=0.8, damping=0.5, wet_level=0.3)
wet = apply_effect(dry, reverb)
display(play_sound(wet, "リバーブあり（大きな部屋）"))

### ディレイを体験する

In [ ]:
from pedalboard import Delay

delay = Delay(delay_seconds=0.3, feedback=0.4, mix=0.5)
wet_delay = apply_effect(dry, delay)
display(play_sound(wet_delay, "ディレイあり（0.3秒のエコー）"))

### コーラスを体験する

In [ ]:
from pedalboard import Chorus

chorus = Chorus(rate_hz=2.0, depth=0.25, mix=0.4)
wet_chorus = apply_effect(dry, chorus)
display(play_sound(wet_chorus, "コーラスあり"))

### 自分で実装してみよう

In [ ]:
def simple_delay(signal, delay_sec=0.3, feedback=0.4, wet_level=0.5,
                 sample_rate=44100):
    """シンプルなディレイエフェクト"""
    delay_samples = int(delay_sec * sample_rate)
    data = signal.data
    output = np.zeros(len(data) + delay_samples * 5)  # エコーの余韻分を確保
    buf = np.zeros(delay_samples)
    idx = 0

    for n in range(len(data)):
        # バッファから遅延信号を取り出す
        delayed = buf[idx]
        # 入力 + フィードバックをバッファに書き込む
        buf[idx] = data[n] + feedback * delayed
        # ドライ音とウェット音を混ぜる
        output[n] = (1.0 - wet_level) * data[n] + wet_level * delayed
        idx = (idx + 1) % delay_samples

    # 残りのバッファを処理（余韻）
    for n in range(len(data), len(output)):
        delayed = buf[idx]
        buf[idx] = feedback * delayed
        output[n] = wet_level * delayed
        idx = (idx + 1) % delay_samples

    return AudioSignal(np.clip(output, -1.0, 1.0), sample_rate)

In [ ]:
dry = create_test_sound()
wet_self = simple_delay(dry, delay_sec=0.3, feedback=0.4, wet_level=0.5)
display(play_sound(dry, "元の音"))
display(play_sound(wet_self, "自作ディレイ（0.3秒, feedback=0.4）"))

### インパルス応答で仕組みを確認する

In [ ]:
import matplotlib.pyplot as plt

# インパルス信号を作成
impulse = np.zeros(44100)  # 1秒
impulse[0] = 1.0
impulse_signal = AudioSignal(impulse, 44100)

# ディレイに通す
delay_out = simple_delay(impulse_signal, delay_sec=0.1, feedback=0.6, wet_level=1.0)

plt.figure(figsize=(12, 4))
t = np.arange(len(delay_out.data)) / 44100
plt.plot(t, delay_out.data)
plt.title("ディレイのインパルス応答（遅延 0.1秒, feedback=0.6）")
plt.xlabel("時間 (秒)")
plt.ylabel("振幅")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### コムフィルタ

In [ ]:
def simple_comb_filter(signal, delay_sec=0.03, feedback=0.7,
                       sample_rate=44100):
    """シンプルなコムフィルタ"""
    delay_samples = int(delay_sec * sample_rate)
    data = signal.data
    buf = np.zeros(delay_samples)
    idx = 0
    output = np.zeros_like(data)

    for n in range(len(data)):
        delayed = buf[idx]
        buf[idx] = data[n] + feedback * delayed
        output[n] = delayed
        idx = (idx + 1) % delay_samples

    return AudioSignal(np.clip(output, -1.0, 1.0), sample_rate)

In [ ]:
dry = create_test_sound()
comb_out = simple_comb_filter(dry, delay_sec=0.03, feedback=0.8)
display(play_sound(dry, "元の音"))
display(play_sound(comb_out, "コムフィルタ通過後（遅延 30ms, feedback=0.8）"))

### なぜ「コム」フィルタなのか

In [ ]:
# コムフィルタのインパルス応答を計算
impulse = np.zeros(44100)
impulse[0] = 1.0
impulse_signal = AudioSignal(impulse, 44100)
comb_impulse = simple_comb_filter(impulse_signal, delay_sec=0.01, feedback=0.7)

# 周波数特性を表示
from scipy.fft import fft

spectrum = np.abs(fft(comb_impulse.data))
freqs = np.arange(len(spectrum)) / len(spectrum) * 44100

plt.figure(figsize=(12, 4))
plt.plot(freqs[:5000], 20 * np.log10(spectrum[:5000] + 1e-10))
plt.title("コムフィルタの周波数特性")
plt.xlabel("周波数 (Hz)")
plt.ylabel("振幅 (dB)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Schroeder リバーブ

In [ ]:
from audio_lib import Reverb as SimpleReverb

dry = create_test_sound()

# audio_lib のリバーブ（Schroeder 方式）
simple_reverb = SimpleReverb(room_size=0.8, damping=0.3, wet_level=0.5)
simple_out = simple_reverb.process(dry)

# pedalboard のリバーブ（プロ品質）
pro_reverb = Reverb(room_size=0.8, damping=0.3, wet_level=0.5)
pro_out = apply_effect(dry, pro_reverb)

display(play_sound(dry, "元の音"))
display(play_sound(simple_out, "audio_lib（Schroeder 方式）"))
display(play_sound(pro_out, "pedalboard（プロ品質）"))

### パラメータと音の関係

In [ ]:
dry = create_test_sound(duration=3.0)

# 遅いコーラス（自然な広がり）
chorus_slow = Chorus(rate_hz=1.5, depth=0.25, mix=0.4)
display(play_sound(apply_effect(dry, chorus_slow), "遅いコーラス（rate=1.5Hz）"))

# 速いコーラス（ビブラート的）
chorus_fast = Chorus(rate_hz=6.0, depth=0.15, mix=0.5)
display(play_sound(apply_effect(dry, chorus_fast), "速いコーラス（rate=6.0Hz）"))

### シンプルなコンプレッサーを実装する

In [ ]:
def simple_compressor(signal, threshold=0.5, ratio=4.0,
                      attack=0.01, release=0.1, sample_rate=44100):
    """シンプルなコンプレッサー"""
    data = signal.data
    output = np.zeros_like(data)
    envelope = 0.0

    # アタック・リリースの係数（指数的な追従）
    attack_coeff = np.exp(-1.0 / (attack * sample_rate))
    release_coeff = np.exp(-1.0 / (release * sample_rate))

    for n in range(len(data)):
        current_level = abs(data[n])

        # エンベロープ追従（音量の包絡線を検出）
        if current_level > envelope:
            envelope += (current_level - envelope) * (1 - attack_coeff)
        else:
            envelope += (current_level - envelope) * (1 - release_coeff)

        # 閾値を超えた部分を圧縮
        if envelope > threshold:
            excess = envelope - threshold
            target = threshold + excess / ratio
            gain = target / envelope
        else:
            gain = 1.0

        output[n] = data[n] * gain

    return AudioSignal(output, sample_rate)

### 動作を確認する

In [ ]:
from audio_lib import note_to_frequency

# 音量にばらつきのある音を作成
notes = [60, 64, 67, 72, 67, 64]  # C4, E4, G4, C5, G4, E4
amplitudes = [0.3, 0.9, 0.5, 1.0, 0.4, 0.8]  # 音量がバラバラ

parts = []
for midi_num, amp in zip(notes, amplitudes):
    freq = note_to_frequency(midi_num)
    wave = sine_wave(freq, 0.5)
    env = adsr(0.5, attack=0.01, decay=0.1, sustain=0.6, release=0.1)
    shaped = AudioSignal(wave.data * env.data * amp, 44100)
    parts.append(shaped.data)

dynamic = AudioSignal(np.concatenate(parts), 44100)
compressed = simple_compressor(dynamic, threshold=0.4, ratio=4.0)

display(play_sound(dynamic, "元の音（音量バラバラ）"))
display(play_sound(compressed, "コンプレッサー適用後"))

### 波形を比較する

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

t = np.arange(len(dynamic.data)) / 44100

axes[0].plot(t, dynamic.data, alpha=0.7)
axes[0].set_title("元の音（音量バラバラ）")
axes[0].set_ylabel("振幅")
axes[0].set_ylim(-1.1, 1.1)
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, compressed.data, alpha=0.7, color='orange')
axes[1].set_title("コンプレッサー適用後")
axes[1].set_xlabel("時間 (秒)")
axes[1].set_ylabel("振幅")
axes[1].set_ylim(-1.1, 1.1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9.7 エフェクトチェーンの順序

In [ ]:
from pedalboard import Pedalboard

dry = create_test_sound(duration=2.0)

# チェーン1: リバーブ → ディレイ
chain1 = Pedalboard([
    Reverb(room_size=0.9, damping=0.2, wet_level=0.7),
    Delay(delay_seconds=0.4, feedback=0.5, mix=0.6),
])
out1 = apply_effect(dry, chain1)

# チェーン2: ディレイ → リバーブ
chain2 = Pedalboard([
    Delay(delay_seconds=0.4, feedback=0.5, mix=0.6),
    Reverb(room_size=0.9, damping=0.2, wet_level=0.7),
])
out2 = apply_effect(dry, chain2)

display(play_sound(out1, "チェーン1: リバーブ → ディレイ"))
display(play_sound(out2, "チェーン2: ディレイ → リバーブ"))